# 03 First model: can we predict machine failure?

Day 1, 13:50 to 14:45 (guided) and 15:00 to 16:15 (independent, Part B).

We will:

1. pick the inputs (features) and the answer (target), dropping anything that leaks the answer
2. split the data into a training set and a test set, keeping the same failure rate in both
3. build a **baseline** that always says "no failure"
4. train a logistic regression and a decision tree
5. judge them on **missed breakdowns** and **false alarms**, not only accuracy

In [ ]:
from pathlib import Path
import pandas as pd

# Works whether Jupyter starts in the repo root or in notebooks/
DATA = next(p for p in [Path("data/raw/ai4i2020.csv"), Path("../data/raw/ai4i2020.csv")] if p.exists())
df = pd.read_csv(DATA)
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
df.shape

## 1. Features and target

- **Target** (`y`): `machine_failure`.
- **Not features:** `udi` and `product_id` (IDs), and the five mode columns (they leak the answer, see notebook 02).
- `type` is text (L/M/H). Models need numbers, so we turn it into three 0/1 columns with `pd.get_dummies` ("one-hot encoding").

In [ ]:
TARGET = "machine_failure"
# TODO: list every column that must NOT be an input (IDs, the 5 mode columns, and the target itself)
DROP = [...]

X = pd.get_dummies(df.drop(columns=DROP), columns=["type"], dtype=int)
y = df[TARGET]
print(X.columns.tolist())
X.head()

## 2. Stratified train/test split

We hide 20% of the rows as a **test set** the model never sees during training: it plays the role of
"next month on the shop floor". `stratify=y` keeps the failure rate the same in both parts.
With only 339 failures, an unlucky random split could leave very few in the test set.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: 20% test, stratified on y, random_state=SEED
X_train, X_test, y_train, y_test = train_test_split(...)

print("train:", X_train.shape, f"failure rate {y_train.mean():.2%}")
print("test: ", X_test.shape, f"failure rate {y_test.mean():.2%}, failures = {y_test.sum()}")

## 3. One scoring function, used for every model

The **confusion matrix** counts four outcomes. In maintenance language:

| | predicted OK | predicted FAIL |
|---|---|---|
| **actually OK** | correct, no action | **false alarm** (wasted technician visit) |
| **actually FAILED** | **missed breakdown** (unplanned downtime) | caught failure |

- **Recall** = caught failures ÷ all real failures. *Of the breakdowns, how many did we see coming?*
- **Precision** = caught failures ÷ all failure alarms. *When we raise an alarm, how often is it real?*
- **F1** = one number that balances precision and recall.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

results = []   # one row per model, so we can compare at the end

def evaluate(name, model, X_te=None, y_te=None):
    X_te = X_test if X_te is None else X_te
    y_te = y_test if y_te is None else y_te
    pred = model.predict(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    row = {
        "model": name,
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "f1": f1_score(y_te, pred, zero_division=0),
        "caught": tp,
        "missed_breakdowns": fn,
        "false_alarms": fp,
    }
    results.append(row)
    print(f"{name}: accuracy {row['accuracy']:.3f} | precision {row['precision']:.3f} | "
          f"recall {row['recall']:.3f} | caught {tp}, missed {fn}, false alarms {fp}")
    return row

## 4. Baseline: always say "no failure"

A baseline is the score to beat. `DummyClassifier(strategy="most_frequent")` always predicts the most common answer.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
evaluate("baseline: always no failure", baseline);

> **The accuracy trap.** About 97% accuracy, and it catches **zero** failures. Accuracy is the wrong
> scoreboard when the thing we care about is rare. From now on we watch **recall** and **missed breakdowns** first.

## 5. Logistic regression

A simple, explainable model: it weighs each input and turns the total into a probability.
It works best when inputs are on similar scales, so we put a `StandardScaler` in front of it in a **pipeline**.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# TODO: fit logreg on the training data, then evaluate it with evaluate("logistic regression", logreg)

## 6. Decision tree

A tree asks yes/no questions about the readings ("torque above 60 Nm?"). `max_depth` limits how many
questions deep it can go, which stops it memorising the training data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# TODO: create a DecisionTreeClassifier with max_depth=6 and random_state=SEED, fit it, evaluate it
tree = ...

### Look inside the tree: its first few questions

In [ ]:
from sklearn.tree import export_text
print(export_text(tree, feature_names=list(X.columns), max_depth=2))

## 7. Confusion matrices side by side

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, model) in zip(axes, [("baseline", baseline), ("logistic regression", logreg), ("decision tree", tree)]):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, display_labels=["ok", "fail"], ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 8. What does each model cost?

Put a price on mistakes. Ask the client (or a maintenance manager) for real numbers; these are placeholders:

- a **missed breakdown** costs unplanned downtime, repairs and scrap
- a **false alarm** costs a technician's inspection

In [ ]:
COST_MISSED = 10_000      # placeholder: one unplanned breakdown
COST_FALSE_ALARM = 500    # placeholder: one unnecessary inspection

table = pd.DataFrame(results).drop_duplicates("model", keep="last").set_index("model")
table["cost"] = table["missed_breakdowns"] * COST_MISSED + table["false_alarms"] * COST_FALSE_ALARM
table.round(3)

**Discuss:** which model would you put on the shop floor today? What would make you trust it more?
Copy the rows above into `results/experiments_by_hand.md`.

---

# Part B: independent practice (15:00 to 16:15, camera off)

Work through B1 to B4. Message the instructor on chat if you are stuck for more than 5 minutes.
**Record every run by hand** in `results/experiments_by_hand.md`: model, settings, features, precision, recall,
missed breakdowns, false alarms. Notice how tedious and error-prone that is. Tomorrow MLflow does it for us.

## B1. Random forest

A random forest trains many decision trees on random samples of the data and lets them vote.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO: RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1), fit, evaluate

## B2. Class weighting for rare failures

`class_weight="balanced"` tells the model that each failure counts about 30 times more than a normal run
during training (because failures are about 30 times rarer). Expect **recall to go up** and **precision to go down**. Why?

In [ ]:
# TODO: repeat logistic regression and random forest with class_weight="balanced"
# Tip for the forest: also try min_samples_leaf=5

## B3. Two engineered features

Add `power_w` and `temp_diff_k` (from notebook 02), re-split with the **same seed**, and retrain your best two models.

In [ ]:
X_fe = X.copy()
# TODO: add the two feature columns to X_fe
X_fe["power_w"] = ...
X_fe["temp_diff_k"] = ...

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fe, y, test_size=0.2, stratify=y, random_state=SEED
)

# TODO: train your best two models on Xf_train / yf_train and evaluate on Xf_test / yf_test
# e.g. evaluate("random forest, balanced + features", forest_fe, Xf_test, yf_test)

Which inputs does the forest rely on most?

In [ ]:
importances = pd.Series(forest_fe.feature_importances_, index=X_fe.columns).sort_values()
importances.plot.barh(title="Random forest feature importance", figsize=(6, 4))
plt.show()

## B4 (stretch). Move the alarm threshold

By default a model raises an alarm when its failure probability is above 0.5. Lowering the threshold
catches more failures but raises more false alarms. This is a **business decision**, not a technical one.

In [ ]:
proba = forest_fe.predict_proba(Xf_test)[:, 1]   # probability of failure for each test row
# TODO: for thresholds 0.2 to 0.7, turn proba into 0/1 predictions and compute precision, recall,
#       missed breakdowns, false alarms and cost. Which threshold gives the lowest cost?

## Final comparison

In [ ]:
final = pd.DataFrame(results).drop_duplicates("model", keep="last").set_index("model")
final["cost"] = final["missed_breakdowns"] * COST_MISSED + final["false_alarms"] * COST_FALSE_ALARM
final.sort_values("cost").round(3)

## Write your evaluation

Open `results/evaluation.md` and fill it in: best model, its precision and recall in plain words,
one maintenance recommendation, and whether you would trust it on the shop floor. Then commit and push:

```bash
git add notebooks/ results/
git commit -m "Day 1: first models and written evaluation"
git push
```